In [3]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme('paper', font_scale=0.8)

import geneinfo.information as gi

plt.rcParams['figure.facecolor'], plt.rcParams['axes.facecolor'] = '#1F1F1F', '#1F1F1F' 

%config InlineBackend.figure_formats = ['svg'] 

In [4]:
common_miss = pd.read_csv('../results/common_missense_human_gnomad.csv')
common_miss

,species,gene_id,gene_name,transcript_id,variant_id,chromosome,genomic_position,ref_allele,alt_allele,aa_position,ref_aa,alt_aa,allele_frequency,hgvsp,hgvsc,consequence,source
0,human,ENSG00000169084,DHRSX,ENST00000334651,X-2221145-C-T,X,2221145,C,T,297,Glu,Lys,0.333643,p.Glu297Lys,c.889G>A,missense_variant,gnomAD_v4
1,human,ENSG00000169084,DHRSX,ENST00000334651,X-2221159-T-C,X,2221159,T,C,292,His,Arg,0.868850,p.His292Arg,c.875A>G,missense_variant,gnomAD_v4
2,human,ENSG00000169084,DHRSX,ENST00000334651,X-2243088-C-G,X,2243088,C,G,247,Val,Leu,0.709305,p.Val247Leu,c.739G>C,missense_variant,gnomAD_v4
3,human,ENSG00000205755,CRLF2,ENST00000400841,X-1193297-T-C,X,1193297,T,C,258,Lys,Arg,0.502598,p.Lys258Arg,c.773A>G,missense_variant,gnomAD_v4
4,human,ENSG00000196433,ASMT,ENST00000381241,X-1632709-T-C,X,1632709,T,C,190,Trp,Arg,0.444030,p.Trp190Arg,c.568T>C,missense_variant,gnomAD_v4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17204,human,ENSG00000114473,IQCG,ENST00000265239,3-197932289-C-G,3,197932289,C,G,177,Asp,His,0.248446,p.Asp177His,c.529G>C,missense_variant,gnomAD_v4
17205,human,ENSG00000114473,IQCG,ENST00000265239,3-197938728-G-T,3,197938728,G,T,112,Ala,Asp,0.248504,p.Ala112Asp,c.335C>A,missense_variant,gnomAD_v4
17206,human,ENSG00000061938,TNK2,ENST00000672887,3-195868079-G-A,3,195868079,G,A,740,Pro,Leu,0.213131,p.Pro740Leu,c.2219C>T,missense_variant,gnomAD_v4
17207,human,ENSG00000122068,FYTTD1,ENST00000241502,3-197768463-G-A,3,197768463,G,A,87,Arg,His,0.835541,p.Arg87His,c.260G>A,missense_variant,gnomAD_v4


In [7]:
all_lof = pd.read_parquet("../results/gnomad_lof/gnomad_lof_all_genes_20251206_003756.parquet")
all_lof


,gene_name,gene_id,transcript_id,variant_id,chrom,position,ref,alt,mutation_type,consequence,...,af_total,af_afr,af_amr,af_asj,af_eas,af_fin,af_nfe,af_oth,af_sas,cds_relative_position
0,A1BG,ENSG00000121410,ENST00000263100,19-58858397-T-C,19,58858397,T,C,Splice acceptor,splice_acceptor_variant,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000009,0.000000,0.000000,Unknown
1,A1BG,ENSG00000121410,ENST00000263100,19-58858718-C-T,19,58858718,C,T,Splice donor,splice_donor_variant,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000033,Unknown
2,A1BG,ENSG00000121410,ENST00000263100,19-58858752-C-A,19,58858752,C,A,In-frame stop codon,stop_gained,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000033,Unknown
3,A1BG,ENSG00000121410,ENST00000263100,19-58858774-C-T,19,58858774,C,T,In-frame stop codon,stop_gained,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000168,0.000000,Unknown
4,A1BG,ENSG00000121410,ENST00000263100,19-58858780-G-GTA,19,58858780,G,GTA,Frameshift,frameshift_variant,...,0.000008,0.000000,0.000000,0.000000,0.000113,0.0,0.000000,0.000000,0.000000,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715938,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185236-G-GC,19,14185236,G,GC,Frameshift,frameshift_variant,...,0.000081,0.000000,0.000000,0.000121,0.000096,0.0,0.000094,0.000240,0.000134,Unknown
715939,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185276-G-A,19,14185276,G,A,Splice donor,splice_donor_variant,...,0.000032,0.000115,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,Unknown
715940,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185514-CCA-C,19,14185514,CCA,C,Splice acceptor,splice_acceptor_variant,...,0.000007,0.000000,0.000041,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,Unknown
715941,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185516-A-AGGG,19,14185516,A,AGGG,Splice acceptor,splice_acceptor_variant,...,0.000007,0.000000,0.000041,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,Unknown


In [37]:
common_lof = common_miss.merge(all_lof, 
                               left_on=['gene_name', 'variant_id'], 
                               right_on=['gene_name', 'variant_id'], how='inner',
                                suffixes=('_com', '_lof')
                                )
common_lof

,species,gene_id_com,gene_name,transcript_id_com,variant_id,chromosome,genomic_position,ref_allele,alt_allele,aa_position,...,af_total,af_afr,af_amr,af_asj,af_eas,af_fin,af_nfe,af_oth,af_sas,cds_relative_position


In [45]:
common_lof = common_miss.merge(all_lof, 
                               left_on=['gene_name', 'genomic_position'], 
                               right_on=['gene_name', 'position'], how='inner',
                                suffixes=('_com', '_lof')
                                )
common_lof#[['gene_name', 'variant_id_com', 'variant_id_lof']]

,species,gene_id_com,gene_name,transcript_id_com,variant_id_com,chromosome,genomic_position,ref_allele,alt_allele,aa_position,...,af_total,af_afr,af_amr,af_asj,af_eas,af_fin,af_nfe,af_oth,af_sas,cds_relative_position
0,human,ENSG00000145975,FAM217A,ENST00000274673,6-4068932-C-T,6,4068932,C,T,431,...,0.000243,0.003148,0.000202,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Unknown
1,human,ENSG00000167664,TMIGD2,ENST00000301272,19-4294626-C-A,19,4294626,C,A,168,...,0.000057,0.000000,0.000413,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Unknown
2,human,ENSG00000116017,ARID3A,ENST00000263620,19-971949-G-A,19,971949,G,A,556,...,0.000028,0.000000,0.000106,0.000000,0.000000,0.000000,0.000021,0.000000,0.000036,Unknown
3,human,ENSG00000099866,MADCAM1,ENST00000215637,19-501719-T-C,19,501719,T,C,240,...,0.000143,0.000000,0.000000,0.000000,0.000000,0.001695,0.000000,0.000000,0.000000,Unknown
4,human,ENSG00000099866,MADCAM1,ENST00000215637,19-501719-T-C,19,501719,T,C,240,...,0.012564,0.003256,0.012821,0.022727,0.010714,0.038983,0.018282,0.024510,0.000000,Unknown
5,human,ENSG00000099866,MADCAM1,ENST00000215637,19-501719-T-C,19,501719,T,C,240,...,0.000286,0.000000,0.000000,0.000000,0.007143,0.000000,0.000000,0.000000,0.000000,Unknown
6,human,ENSG00000105289,TJP3,ENST00000541714,19-3750617-T-C,19,3750617,T,C,898,...,0.000070,0.000000,0.000089,0.000000,0.000000,0.000000,0.000082,0.000167,0.000136,Unknown
7,human,ENSG00000106714,CNTNAP3,ENST00000297668,9-39078846-A-G,9,39078846,A,G,1173,...,0.000032,0.000000,0.000000,0.000000,0.000644,0.000000,0.000000,0.000000,0.000000,Unknown
8,human,ENSG00000106714,CNTNAP3,ENST00000297668,9-39088492-T-A,9,39088492,T,A,1051,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Unknown
9,human,ENSG00000184956,MUC6,ENST00000421673,11-1016959-T-C,11,1016959,T,C,1948,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Unknown


In [41]:
gi.gene_info(common_lof.gene_name.unique().tolist())

**Symbol:** **_FAM217A_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** C6orf146  
*family with sequence similarity 217 member A*  
**Human genomic position:** 6:4049434-4087344 (hg38), 6:4049668-4087578 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=FAM217A)  


 ----

**Symbol:** **_TMIGD2_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** CD28H, IGPR-1, IGPR1  
*transmembrane and immunoglobulin domain containing 2*  
**Summary:** Enables coreceptor activity. Involved in positive regulation of T cell activation; positive regulation of angiogenesis; and positive regulation of cytokine production. Predicted to be located in plasma membrane. Predicted to be integral component of membrane. Predicted to be active in extracellular space. [provided by Alliance of Genome Resources, Apr 2022]  
**Human genomic position:** 19:4292227-4302431 (hg38), 19:4292229-4302428 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=TMIGD2)  


 ----

**Symbol:** **_ARID3A_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** BRIGHT, DRIL1, DRIL3, E2FBP1  
*AT-rich interaction domain 3A*  
**Summary:** This gene encodes a member of the ARID (AT-rich interaction domain) family of DNA binding proteins. It was found by homology to the Drosophila dead ringer gene, which is important for normal embryogenesis. Other ARID family members have roles in embryonic patterning, cell lineage gene regulation, cell cycle control, transcriptional regulation, and possibly in chromatin structure modification. [provided by RefSeq, Jul 2008].  
**Human genomic position:** 19:924528-975939 (hg38), 19:925781-975939 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=ARID3A)  


 ----

**Symbol:** **_MADCAM1_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** MACAM1  
*mucosal vascular addressin cell adhesion molecule 1*  
**Summary:** The protein encoded by this gene is an endothelial cell adhesion molecule that interacts preferentially with the leukocyte beta7 integrin LPAM-1 (alpha4beta7), L-selectin, and VLA-4 (alpha4beta1) on myeloid cells to direct leukocytes into mucosal and inflamed tissues. It is a member of the immunoglobulin family and is similar to ICAM1 and VCAM1. At least seven alternatively spliced transcripts encoding different protein isoforms have been found for this gene, but the full-length nature of some variants has not been determined. [provided by RefSeq, Jul 2008].  
**Human genomic position:** 19:489176-505343 (hg38), 19:489176-505347 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=MADCAM1)  


 ----

**Symbol:** **_TJP3_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** ZO-3, ZO3  
*tight junction protein 3*  
**Summary:** The protein encoded by this gene is a member of the membrane-associated guanylate kinase-like (MAGUK) protein family which is characterized by members having multiple PDZ domains, a single SH3 domain, and a single guanylate kinase-like (GUK)-domain. In addition, members of the zonula occludens protein subfamily have an acidic domain, a basic arginine-rich region, and a proline-rich domain. The protein encoded by this gene plays a role in the linkage between the actin cytoskeleton and tight-junctions and also sequesters cyclin D1 at tight junctions during mitosis. Alternative splicing results in multiple transcript variants encoding distinct isoforms. This gene has a partial pseudogene on chromosome 1. [provided by RefSeq, May 2012].  
**Human genomic position:** 19:3708356-3750813 (hg38), 19:3708107-3750811 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=TJP3)  


 ----

...

**Symbol:** **_DRD4_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** D4DR  
*dopamine receptor D4*  
**Summary:** This gene encodes the D4 subtype of the dopamine receptor. The D4 subtype is a G-protein coupled receptor which inhibits adenylyl cyclase. It is a target for drugs which treat schizophrenia and Parkinson disease. Mutations in this gene have been associated with various behavioral phenotypes, including autonomic nervous system dysfunction, attention deficit/hyperactivity disorder, and the personality trait of novelty seeking. This gene contains a polymorphic number (2-10 copies) of tandem 48 nt repeats; the sequence shown contains four repeats. [provided by RefSeq, Jul 2008].  
**Human genomic position:** 11:637269-640706, HSCHR11_1_CTG8:167166-170577 (hg38), 11:637293-640706 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=DRD4)  


 ----

**Symbol:** **_PNPLA2_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** 1110001C14Rik, ATGL, FP17548, PEDF-R, TTS-2.2, TTS2, iPLA2zeta  
*patatin like domain 2, triacylglycerol lipase*  
**Summary:** This gene encodes an enzyme which catalyzes the first step in the hydrolysis of triglycerides in adipose tissue. Mutations in this gene are associated with neutral lipid storage disease with myopathy. [provided by RefSeq, Jul 2010].  
**Human genomic position:** 11:818914-825573 (hg38), 11:818902-825573 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=PNPLA2)  


 ----

**Symbol:** **_CDHR5_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** MLPCDH, MU-PCDH, MUCDHL, MUPCDH  
*cadherin related family member 5*  
**Summary:** This gene is a novel mucin-like gene that is a member of the cadherin superfamily. While encoding nonpolymorphic tandem repeats rich in proline, serine and threonine similar to mucin proteins, the gene also contains sequence encoding calcium-binding motifs found in all cadherins. The role of the hybrid extracellular region and the specific function of this protein have not yet been determined. Alternatively spliced transcript variants encoding different isoforms have been described. [provided by RefSeq, Jan 2010].  
**Human genomic position:** 11:616405-626078, HSCHR11_1_CTG8:146464-155977 (hg38), 11:616565-626078 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=CDHR5)  


 ----

**Symbol:** **_PLEKHA6_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** PEPP-3, PEPP3  
*pleckstrin homology domain containing A6*  
**Human genomic position:** 1:204218851-204378214 (hg38), 1:204187979-204346793 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=PLEKHA6)  


 ----

**Symbol:** **_OSMR_** (protein-coding) &nbsp; &nbsp; &nbsp; &nbsp; **Aliases:** IL-31R-beta, IL-31RB, OSMRB, OSMRbeta, PLCA1  
*oncostatin M receptor*  
**Summary:** This gene encodes a member of the type I cytokine receptor family. The encoded protein heterodimerizes with interleukin 6 signal transducer to form the type II oncostatin M receptor and with interleukin 31 receptor A to form the interleukin 31 receptor, and thus transduces oncostatin M and interleukin 31 induced signaling events. Mutations in this gene have been associated with familial primary localized cutaneous amyloidosis. Alternatively spliced transcript variants encoding different isoforms have been found for this gene. [provided by RefSeq, Dec 2009].  
**Human genomic position:** 5:38845852-38945596 (hg38), 5:38845960-38945698 (hg19)  
[Gene card](https://www.genecards.org/cgi-bin/carddisp.pl?gene=OSMR)  


 ----

In [34]:
df = pd.read_parquet("../data/alpha_missense_hg38.parquet")
regex = re.compile(r'(.)(\d+)(.)')
ref, pos, alt = zip(*[regex.match(x).groups() for x in df.protein_variant])
df['ref_aa'] = ref
df['pos'] = list(map(int, pos))
df['alt_aa'] = alt
df

,uniprot_id,protein_variant,am_pathogenicity,am_class,ref_aa,pos,alt_aa
0,A0A024R1R8,M1A,0.4673,ambiguous,M,1,A
1,A0A024R1R8,M1C,0.3828,ambiguous,M,1,C
2,A0A024R1R8,M1D,0.8267,pathogenic,M,1,D
3,A0A024R1R8,M1E,0.5236,ambiguous,M,1,E
4,A0A024R1R8,M1F,0.2753,benign,M,1,F
...,...,...,...,...,...,...,...
216175346,X6R8D5,S127R,0.6837,pathogenic,S,127,R
216175347,X6R8D5,S127T,0.0885,benign,S,127,T
216175348,X6R8D5,S127V,0.4471,ambiguous,S,127,V
216175349,X6R8D5,S127W,0.5592,ambiguous,S,127,W


In [ ]:
uniprot_ids = df.uniprot_id.unique()
uniprot_ids

In [ ]:
uniprot2hgcn = {}
for x in tqdm(df.uniprot_id.unique()):
    try:
        uniprot2hgcn[x] = gi.hgcn_symbol(x)
    except gi.NotFound:
        uniprot2hgcn[x] = None

  0%|          | 0/20516 [00:00<?, ?it/s]

NotFound: 

In [ ]:
common_miss.join(df, on=['uniprot_id', 'pos', 'ref_aa', 'alt_aa'], how='left')

In [ ]:
uniprot_id = 'Q8NHH1'
df = pd.read_parquet("../data/alpha_missense_hg38.parquet", filters=[("uniprot_id", "==", f"{uniprot_id}")])
regex.match(r'(.)(\d+)(.)').groups() df.protein_variant
df

,uniprot_id,protein_variant,am_pathogenicity,am_class
0,Q8NHH1,M1A,0.3139,benign
1,Q8NHH1,M1C,0.4108,ambiguous
2,Q8NHH1,M1D,0.8912,pathogenic
3,Q8NHH1,M1E,0.7156,pathogenic
4,Q8NHH1,M1F,0.2341,benign
...,...,...,...,...
15195,Q8NHH1,S800R,0.2823,benign
15196,Q8NHH1,S800T,0.0887,benign
15197,Q8NHH1,S800V,0.1524,benign
15198,Q8NHH1,S800W,0.1835,benign


In [ ]:
pd.read_csv('../results/gnomad_lof/gnomad_lof_all_genes_20251206_003756.csv')